# EchoCare-CLIP Training Notebook

This notebook trains a **CLIP-style contrastive model** that aligns echocardiography images (encoded by the pretrained **EchoCare SwinTransformer**) with clinical text descriptions (encoded by the **CLIP text encoder**).

## Architecture overview
```
Image (B, 3, 256, 256)
    → EchoCare SwinTransformer    → multi-scale features
    → Global Average Pool (GAP)   → (B, 2048)
    → MLP projection head         → (B, projection_dim)
    → L2 normalize                → image_emb (B, projection_dim)

Text (list of B strings)
    → CLIP tokenizer              → (B, 77) token ids
    → CLIP text encoder           → (B, 512)
    → MLP projection head         → (B, projection_dim)
    → L2 normalize                → text_emb (B, projection_dim)

Loss: InfoNCE (symmetric cross-entropy on B×B cosine similarity matrix)
```

## Design choices
- **Image encoder**: EchoCare SwinTransformer (`feature_size=128, depths=[2,2,18,2]`) pretrained with self-supervised MAE on echo data. We use its deepest feature map and apply GAP.
- **Text encoder**: `openai/clip-vit-base-patch32` text tower (~63 M params). Chosen because (a) it is purpose-built for image–text alignment, (b) it is compact relative to larger language models, and (c) its embedding space is already semantically rich. We freeze its backbone and only train a projection head.
- **Shared space**: `projection_dim=256`. Both modalities are L2-normalized before computing cosine similarity.
- **Temperature**: Learnable `log_temperature`, initialized at `log(0.07)` following the original CLIP paper.

## Step 0 — Install dependencies

In [1]:
# Install required packages.
# - monai:        provides the SwinTransformer used by EchoCare
# - transformers: provides CLIPTextModel and CLIPTokenizer
# - einops:       needed internally by MONAI's SwinTransformer
!pip install monai torch torchvision transformers einops --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 61.2 MB/s eta 0:00:00


## Step 1 — Imports

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from torch.utils.data import Dataset, DataLoader

# MONAI SwinTransformer — the backbone used by EchoCare
from monai.networks.nets.swin_unetr import SwinTransformer

# Hugging Face CLIP text tower and tokenizer
from transformers import CLIPTokenizer, CLIPTextModel

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


PyTorch version : 2.10.0+cu128
CUDA available  : True


## Step 2 — Configuration

All hyperparameters and paths live in a single `config` dict for easy modification.

In [3]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Image encoder (EchoCare SwinTransformer)
#   feature_size=128 → stage channel dims: 128, 256, 512, 1024, 2048
#   Input image: (B, 3, 256, 256)
#   Deepest feature map before GAP: (B, 2048, 8, 8)
#   After GAP: (B, 2048)
#
# Text encoder (CLIP, openai/clip-vit-base-patch32)
#   Input token ids: (B, 77)  [CLIP's fixed max context length]
#   Pooled text output: (B, 512)
#
# Shared projection space: projection_dim=256
# ───────────────────────────────────────────────────────────────────────────────

config = {
    # ── EchoCare image encoder ──────────────────────────────────────────────
    "feature_size"        : 128,          # SwinTransformer base channel dim
    "in_channels"         : 3,            # RGB (echo frames exported as 3-ch)
    "image_size"          : 256,          # H = W of input images
    "image_embed_dim"     : 2048,         # deepest Swin stage output channels
    "pretrained_checkpoint": "echocare_encoder.pth",  # path to EchoCare weights

    # ── Text encoder ────────────────────────────────────────────────────────
    "clip_model_name"     : "openai/clip-vit-base-patch32",
    "text_embed_dim"      : 512,          # CLIP text encoder hidden size
    "max_seq_len"         : 77,           # CLIP tokenizer max length

    # ── Shared latent space ─────────────────────────────────────────────────
    "projection_dim"      : 256,          # dim after projection heads

    # ── Training ────────────────────────────────────────────────────────────
    "batch_size"          : 4,
    "num_epochs"          : 3,            # small for quick end-to-end check
    "lr"                  : 1e-4,
    "weight_decay"        : 1e-4,
    "init_temperature"    : 0.07,         # CLIP default starting temperature

    # ── Dummy data ──────────────────────────────────────────────────────────
    "num_dummy_samples"   : 16,           # total samples in fake dataset
}

print("Configuration loaded:")
for k, v in config.items():
    print(f"  {k:25s}: {v}")

Configuration loaded:
  feature_size             : 128
  in_channels              : 3
  image_size               : 256
  image_embed_dim          : 2048
  pretrained_checkpoint    : echocare_encoder.pth
  clip_model_name          : openai/clip-vit-base-patch32
  text_embed_dim           : 512
  max_seq_len              : 77
  projection_dim           : 256
  batch_size               : 4
  num_epochs               : 3
  lr                       : 0.0001
  weight_decay             : 0.0001
  init_temperature         : 0.07
  num_dummy_samples        : 16


## Step 3 — Dummy Dataset and DataLoader

We create a synthetic dataset of `(image, text)` pairs so the entire pipeline can be verified before real data is available.

**When connecting real data**, replace `DummyEchoDataset.__getitem__` with:
1. Load a DICOM / PNG echo frame → resize to `(3, 256, 256)` float tensor in `[0, 1]`.
2. Return the associated clinical report string.

In [4]:
# ── Dummy clinical text strings ────────────────────────────────────────────────
# In production these would be structured reports or label strings
# associated with each echo study.
DUMMY_TEXTS = [
    "Normal left ventricular function with no wall motion abnormalities.",
    "Mild mitral regurgitation with preserved ejection fraction.",
    "Severe aortic stenosis with reduced cardiac output.",
    "Dilated cardiomyopathy with global hypokinesis.",
    "Hypertrophic cardiomyopathy with outflow tract obstruction.",
    "Pericardial effusion with signs of cardiac tamponade.",
    "Normal right ventricular size and systolic function.",
    "Tricuspid regurgitation with elevated right heart pressures.",
]


class DummyEchoDataset(Dataset):
    """
    Synthetic (image, text) dataset for pipeline verification.

    Each sample:
      image : torch.Tensor  shape (3, H, W) = (3, 256, 256),  float32 in [0,1]
                            Simulates a normalised echocardiography frame.
      text  : str           A clinical description string.
                            In production: corresponding structured report.

    DataLoader collation:
      images : (B, 3, 256, 256)   — stacked by PyTorch default collate
      texts  : list[str] of len B — kept as strings; tokenised later
    """

    def __init__(self, num_samples: int, image_size: int = 256):
        self.num_samples = num_samples
        self.image_size  = image_size

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # ── Image ──────────────────────────────────────────────────────────
        # Random noise in [0, 1] mimics a normalised echo frame.
        # Shape: (3, 256, 256)  → channels-first, matching torchvision convention.
        image = torch.rand(3, self.image_size, self.image_size)

        # ── Text ───────────────────────────────────────────────────────────
        # Cycle through preset descriptions so each sample has a unique label.
        text = DUMMY_TEXTS[idx % len(DUMMY_TEXTS)]

        return image, text


# ── Build dataset and dataloader ───────────────────────────────────────────────
dataset = DummyEchoDataset(
    num_samples=config["num_dummy_samples"],
    image_size=config["image_size"],
)

dataloader = DataLoader(
    dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=0,   # 0 for compatibility; increase for real data
)

# ── Sanity check shapes ────────────────────────────────────────────────────────
sample_images, sample_texts = next(iter(dataloader))
print(f"Batch image shape : {sample_images.shape}")  # Expected: (4, 3, 256, 256)
print(f"Batch texts       : {sample_texts}")          # Expected: list of 4 strings
print(f"Dataset length    : {len(dataset)}")

Batch image shape : torch.Size([4, 3, 256, 256])
Batch texts       : ('Tricuspid regurgitation with elevated right heart pressures.', 'Pericardial effusion with signs of cardiac tamponade.', 'Normal right ventricular size and systolic function.', 'Hypertrophic cardiomyopathy with outflow tract obstruction.')
Dataset length    : 16


## Step 4 — EchoCare Image Encoder

Wraps the pretrained SwinTransformer with a two-layer MLP projection head.

**SwinTransformer output stages** (with `feature_size=128`, input `256×256`):

| Stage | Channels | Spatial size |
|-------|----------|--------------|
| 0     | 128      | 128×128      |
| 1     | 256      | 64×64        |
| 2     | 512      | 32×32        |
| 3     | 1024     | 16×16        |
| 4     | **2048** | **8×8**      |

We use **stage 4** (deepest, most semantic) and apply **Global Average Pool** to get a `(B, 2048)` vector, then project to `(B, projection_dim)`.

In [5]:
class EchoCareImageEncoder(nn.Module):
    """
    EchoCare SwinTransformer backbone + MLP projection head.

    Data shapes through the forward pass
    -------------------------------------
    Input image          : (B, 3, 256, 256)
    Swin stage 0         : (B, 128,  128, 128)
    Swin stage 1         : (B, 256,   64,  64)
    Swin stage 2         : (B, 512,   32,  32)
    Swin stage 3         : (B, 1024,  16,  16)
    Swin stage 4 (deep)  : (B, 2048,   8,   8)
    After GAP            : (B, 2048)
    After MLP projection : (B, projection_dim)
    After L2 norm        : (B, projection_dim)  ← output
    """

    def __init__(
        self,
        feature_size   : int  = 128,
        in_channels    : int  = 3,
        projection_dim : int  = 256,
        use_checkpoint : bool = True,
        freeze_encoder : bool = False,
    ):
        super().__init__()

        # ── EchoCare SwinTransformer backbone ─────────────────────────────
        # Exact same config as echocare_test.ipynb to ensure weight compatibility.
        # spatial_dims=2  →  2-D (frame-level, not volumetric)
        # use_v2=True     →  SwinV2 improvements (log-spaced relative position bias)
        self.encoder = SwinTransformer(
            in_chans      = in_channels,
            embed_dim     = feature_size,        # 128
            window_size   = [8, 8],
            patch_size    = [2, 2],
            depths        = [2, 2, 18, 2],
            num_heads     = [4, 8, 16, 32],
            mlp_ratio     = 4.0,
            qkv_bias      = True,
            use_checkpoint= use_checkpoint,
            spatial_dims  = 2,
            use_v2        = True,
        )

        # ── Compute deepest feature dim ───────────────────────────────────
        # Each Swin stage doubles channel count; 4 downsampling stages:
        # 128 → 256 → 512 → 1024 → 2048
        num_downsample   = 4          # len(depths) - 1, since last stage is bottleneck
        encoder_out_dim  = feature_size * (2 ** num_downsample)  # 128 * 16 = 2048

        # ── Projection head ───────────────────────────────────────────────
        # Two-layer MLP with ReLU, following MoCo v3 / CLIP convention.
        # Maps 2048-dim image embedding → projection_dim for alignment.
        self.projection = nn.Sequential(
            nn.Linear(encoder_out_dim, encoder_out_dim),
            nn.ReLU(),
            nn.Linear(encoder_out_dim, projection_dim),
        )

        # ── Optional backbone freeze ──────────────────────────────────────
        # Freeze if the pretrained backbone is strong and we only want to
        # learn the projection. Set to False to allow full fine-tuning.
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

    def load_pretrained(self, checkpoint_path: str, device: str = "cpu"):
        """
        Load EchoCare self-supervised pretrained weights.

        The checkpoint stores a flat state_dict of SwinTransformer weights
        plus an extra 'mask_token' key from MAE pretraining that does not
        exist in the plain SwinTransformer, so we pop it before loading.
        """
        state_dict = torch.load(checkpoint_path, map_location=device)
        state_dict.pop("mask_token", None)   # MAE artefact — not in SwinTransformer
        self.encoder.load_state_dict(state_dict, strict=True)
        print(f"Loaded EchoCare pretrained weights from '{checkpoint_path}'")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (B, 3, 256, 256)  — normalised echo frames
        returns: (B, projection_dim) L2-normalised image embeddings
        """
        # Multi-scale feature extraction
        # feats[k] shape: see docstring table above
        feats = self.encoder(x)

        # Use the deepest (most semantic) feature map
        deep = feats[-1]              # (B, 2048, 8, 8)

        # Global Average Pooling over spatial dimensions
        emb  = deep.mean(dim=(2, 3)) # (B, 2048)

        # Project to shared latent space
        emb  = self.projection(emb)  # (B, projection_dim)

        # L2 normalise so dot product == cosine similarity
        emb  = F.normalize(emb, dim=-1)  # (B, projection_dim)

        return emb

## Step 5 — Text Encoder

**Choice: CLIP text encoder (`openai/clip-vit-base-patch32`)**

Rationale for this choice over alternatives:

| Option | Pros | Cons |
|--------|------|------|
| **CLIP text encoder** (chosen) | Purpose-built for image–text alignment; compact (~63 M); already meaningful token embeddings | Not domain-adapted to medical text |
| GPT-2 | Flexible; widely supported | Causal LM, not ideal for embedding; needs pooling heuristic |
| DistilBERT | Very small; fast | No pretraining on image–text pairs |
| PubMedBERT | Medical domain | Larger; no prior image–text alignment |

We **freeze** the CLIP text encoder backbone and only train its projection head. This prevents catastrophic forgetting of the pretrained text representations while the image encoder adapts.

In [6]:
class CLIPTextEncoder(nn.Module):
    """
    CLIP text tower + MLP projection head.

    Data shapes through the forward pass
    -------------------------------------
    Input token ids    : (B, 77)              — CLIP fixed max length
    Attention mask     : (B, 77)
    CLIP pooler output : (B, 512)             — [EOS] token embedding
    After MLP proj     : (B, projection_dim)
    After L2 norm      : (B, projection_dim)  ← output

    Note: CLIP uses the embedding at the position of the [EOS] token
    (highest-indexed non-padding token) as the sentence representation.
    `pooler_output` from CLIPTextModel returns exactly this.
    """

    def __init__(
        self,
        clip_model_name     : str  = "openai/clip-vit-base-patch32",
        projection_dim      : int  = 256,
        freeze_text_encoder : bool = True,
    ):
        super().__init__()

        # ── Pretrained CLIP text tower ────────────────────────────────────
        # CLIPTextModel is the text-only part of the full CLIP model.
        # hidden_size = 512 for clip-vit-base-patch32.
        self.text_encoder = CLIPTextModel.from_pretrained(clip_model_name)
        text_out_dim      = self.text_encoder.config.hidden_size  # 512

        # ── Freeze backbone (recommended starting point) ──────────────────
        # Only the projection head is updated during early training.
        # Unfreeze text_encoder if domain shift from general CLIP is large.
        if freeze_text_encoder:
            for param in self.text_encoder.parameters():
                param.requires_grad = False

        # ── Projection head ───────────────────────────────────────────────
        # Same two-layer MLP structure as the image encoder's projection head
        # for symmetry.
        self.projection = nn.Sequential(
            nn.Linear(text_out_dim, text_out_dim),
            nn.ReLU(),
            nn.Linear(text_out_dim, projection_dim),
        )

    def forward(
        self,
        input_ids     : torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        input_ids      : (B, 77)  LongTensor of token ids
        attention_mask : (B, 77)  1 for real tokens, 0 for padding
        returns        : (B, projection_dim) L2-normalised text embeddings
        """
        # CLIP text encoding
        # pooler_output: embedding of the [EOS] (end-of-sequence) token,
        # which aggregates the full sequence representation in CLIP.
        outputs  = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        text_emb = outputs.pooler_output   # (B, 512)

        # Project to shared latent space
        text_emb = self.projection(text_emb)  # (B, projection_dim)

        # L2 normalise
        text_emb = F.normalize(text_emb, dim=-1)  # (B, projection_dim)

        return text_emb

## Step 6 — EchoCare-CLIP: Combined CLIP-style Model

Combines both encoders and implements the **InfoNCE (symmetric cross-entropy) loss**:

```
logits_per_image[i, j] = sim(image_i, text_j) / temperature
                                             ↑ (B × B matrix)
Diagonal = positive pairs (same study).
Loss = 0.5 * (CE_rows + CE_cols),  labels = [0, 1, 2, ..., B-1]
```

In [7]:
class EchoCare_CLIP(nn.Module):
    """
    CLIP-style contrastive model:
      EchoCare image encoder  ↕  alignment  ↕  CLIP text encoder

    Training objective
    ------------------
    Given a batch of B matched (image, text) pairs:
      1. Compute image_emb: (B, projection_dim)  — L2 normalised
      2. Compute text_emb : (B, projection_dim)  — L2 normalised
      3. Similarity matrix: logits = image_emb @ text_emb.T / temperature
                            shape  : (B, B)
      4. InfoNCE (symmetric cross-entropy):
           loss = 0.5 * [CE(logits, labels) + CE(logits.T, labels)]
           where labels = [0, 1, ..., B-1]  (diagonal = positives)

    Temperature
    -----------
    Stored as log_temperature (log scale) for numerical stability.
    Actual temperature = exp(log_temperature), learnable parameter.
    Initialised to log(0.07) following the original CLIP paper.
    """

    def __init__(
        self,
        image_encoder   : EchoCareImageEncoder,
        text_encoder    : CLIPTextEncoder,
        init_temperature: float = 0.07,
    ):
        super().__init__()
        self.image_encoder  = image_encoder
        self.text_encoder   = text_encoder

        # Learnable temperature (log scale)
        self.log_temperature = nn.Parameter(
            torch.log(torch.tensor(init_temperature))
        )

    # ── Convenience encode methods for inference ───────────────────────────
    def encode_image(self, images: torch.Tensor) -> torch.Tensor:
        """images: (B, 3, 256, 256) → (B, projection_dim)"""
        return self.image_encoder(images)

    def encode_text(
        self,
        input_ids     : torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """input_ids: (B, 77) → (B, projection_dim)"""
        return self.text_encoder(input_ids, attention_mask)

    def forward(
        self,
        images        : torch.Tensor,
        input_ids     : torch.Tensor,
        attention_mask: torch.Tensor | None = None,
    ):
        """
        Full forward pass (training).

        Args
        ----
        images         : (B, 3, 256, 256)
        input_ids      : (B, 77)             token ids from CLIPTokenizer
        attention_mask : (B, 77)             padding mask

        Returns
        -------
        loss      : scalar  InfoNCE contrastive loss
        image_emb : (B, projection_dim)  L2-normalised image embeddings
        text_emb  : (B, projection_dim)  L2-normalised text embeddings
        """
        # Encode both modalities
        image_emb = self.encode_image(images)                       # (B, D)
        text_emb  = self.encode_text(input_ids, attention_mask)     # (B, D)

        # Current temperature (positive scalar)
        temperature = self.log_temperature.exp()   # scalar

        # Pairwise cosine similarity matrix, scaled by 1/temperature
        # Both embeddings are L2-normalised → dot product == cosine similarity
        logits_per_image = (image_emb @ text_emb.T) / temperature   # (B, B)
        logits_per_text  = logits_per_image.T                        # (B, B)

        # Ground-truth: sample i is paired with text i (diagonal)
        labels = torch.arange(image_emb.size(0), device=image_emb.device)  # (B,)

        # Symmetric InfoNCE loss
        loss_i2t = F.cross_entropy(logits_per_image, labels)
        loss_t2i = F.cross_entropy(logits_per_text,  labels)
        loss     = (loss_i2t + loss_t2i) / 2

        return loss, image_emb, text_emb

## Step 7 — Initialise Models and Load Pretrained Weights

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Image encoder ─────────────────────────────────────────────────────────────
# use_checkpoint=False: disable gradient checkpointing for the dummy run
#   (checkpointing trades compute for memory; enable it for large batches)
# freeze_encoder=False: allow full fine-tuning; set True if GPU memory is tight
image_encoder = EchoCareImageEncoder(
    feature_size   = config["feature_size"],
    in_channels    = config["in_channels"],
    projection_dim = config["projection_dim"],
    use_checkpoint = False,
    freeze_encoder = False,
)

# Load pretrained EchoCare weights.
# Expected checkpoint path: "echocare_encoder.pth"
# Falls back to random init if the file is not found (fine for code verification).
try:
    image_encoder.load_pretrained(
        config["pretrained_checkpoint"],
        device=str(device),
    )
except FileNotFoundError:
    print(f"[WARNING] Checkpoint not found at '{config['pretrained_checkpoint']}'.")
    print("[WARNING] Continuing with random initialisation for code verification.")

# ── Text encoder ──────────────────────────────────────────────────────────────
# freeze_text_encoder=True: CLIP backbone is frozen; only projection is trained.
# This is the recommended starting point to avoid destroying CLIP's alignment.
text_encoder = CLIPTextEncoder(
    clip_model_name     = config["clip_model_name"],
    projection_dim      = config["projection_dim"],
    freeze_text_encoder = True,
)

# ── Tokenizer (needed before training loop) ───────────────────────────────────
tokenizer = CLIPTokenizer.from_pretrained(config["clip_model_name"])

# ── Combined model ────────────────────────────────────────────────────────────
model = EchoCare_CLIP(
    image_encoder    = image_encoder,
    text_encoder     = text_encoder,
    init_temperature = config["init_temperature"],
).to(device)

# ── Parameter counts ──────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"  (CLIP text encoder backbone is frozen → smaller effective model)")

Device: cuda
[WARNING] Checkpoint not found at 'echocare_encoder.pth'.
[WARNING] Continuing with random initialisation for code verification.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]


Total parameters     : 188,502,521
Trainable parameters : 125,336,569
  (CLIP text encoder backbone is frozen → smaller effective model)


## Step 8 — Text Tokenisation Helper

In [9]:
def tokenize_texts(
    texts      : list[str],
    tokenizer  : CLIPTokenizer,
    max_length : int = 77,
    device     : torch.device | str = "cpu",
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Tokenise a list of strings with the CLIP tokenizer.

    CLIP uses a fixed context window of 77 tokens (including [SOS] and [EOS]).
    Sequences shorter than 77 are right-padded; longer ones are truncated.

    Args
    ----
    texts      : list of B strings
    tokenizer  : CLIPTokenizer instance
    max_length : 77 (CLIP default)
    device     : target device

    Returns
    -------
    input_ids      : (B, 77)  LongTensor
    attention_mask : (B, 77)  LongTensor  (1 = real token, 0 = padding)
    """
    encoded = tokenizer(
        texts,
        padding   = "max_length",
        max_length = max_length,
        truncation = True,
        return_tensors = "pt",
    )
    return (
        encoded["input_ids"].to(device),
        encoded["attention_mask"].to(device),
    )


# ── Quick tokeniser sanity check ──────────────────────────────────────────────
test_texts = ["Normal cardiac function.", "Severe aortic stenosis."]
ids, mask  = tokenize_texts(test_texts, tokenizer, device=device)
print(f"input_ids shape      : {ids.shape}"     )  # Expected: (2, 77)
print(f"attention_mask shape : {mask.shape}"    )  # Expected: (2, 77)
print(f"Non-padding tokens   : {mask.sum(dim=1).tolist()}")  # token counts per string

input_ids shape      : torch.Size([2, 77])
attention_mask shape : torch.Size([2, 77])
Non-padding tokens   : [6, 8]


## Step 9 — Training Loop

In [10]:
# ── Optimiser ─────────────────────────────────────────────────────────────────
# AdamW with weight decay is standard for transformer fine-tuning.
# filter(requires_grad) skips the frozen CLIP text backbone.
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr           = config["lr"],
    weight_decay = config["weight_decay"],
)

# ── Learning-rate schedule note ───────────────────────────────────────────────
# For real training: use a cosine decay schedule with linear warmup
# (torch.optim.lr_scheduler.CosineAnnealingLR or transformers.get_cosine_schedule_with_warmup).
# Omitted here to keep the end-to-end test minimal.


def train_one_epoch(
    model      : EchoCare_CLIP,
    dataloader : DataLoader,
    optimizer  : torch.optim.Optimizer,
    tokenizer  : CLIPTokenizer,
    device     : torch.device,
    epoch      : int,
) -> float:
    """
    Run one full training epoch.

    Batch data flow
    ---------------
    images (B, 3, 256, 256)  →  image_emb  (B, projection_dim)
    texts  list[str] of B    →  text_emb   (B, projection_dim)
    loss: scalar InfoNCE value

    Returns average loss over all batches.
    """
    model.train()
    total_loss = 0.0

    for batch_idx, (images, texts) in enumerate(dataloader):
        # images : (B, 3, 256, 256)
        # texts  : list of B strings
        images = images.to(device)

        # Tokenise text on the fly
        # input_ids      : (B, 77)
        # attention_mask : (B, 77)
        input_ids, attention_mask = tokenize_texts(
            texts, tokenizer, device=device
        )

        optimizer.zero_grad()

        # Forward pass
        # loss      : scalar
        # image_emb : (B, projection_dim)
        # text_emb  : (B, projection_dim)
        loss, image_emb, text_emb = model(images, input_ids, attention_mask)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        print(
            f"  [Epoch {epoch+1}] Batch {batch_idx+1}/{len(dataloader)}"
            f" | Loss: {loss.item():.4f}"
            f" | Temp: {model.log_temperature.exp().item():.4f}"
            f" | img_emb: {tuple(image_emb.shape)}"
            f" | txt_emb: {tuple(text_emb.shape)}"
        )

    return total_loss / len(dataloader)


# ── Run training ──────────────────────────────────────────────────────────────
print("Starting training...\n")
for epoch in range(config["num_epochs"]):
    avg_loss = train_one_epoch(
        model, dataloader, optimizer, tokenizer, device, epoch
    )
    print(f">>> Epoch {epoch+1}/{config['num_epochs']}  Avg Loss: {avg_loss:.4f}\n")

print("Training complete.")

Starting training...

  [Epoch 1] Batch 1/4 | Loss: 1.4526 | Temp: 0.0700 | img_emb: (4, 256) | txt_emb: (4, 256)
  [Epoch 1] Batch 2/4 | Loss: 1.5140 | Temp: 0.0700 | img_emb: (4, 256) | txt_emb: (4, 256)
  [Epoch 1] Batch 3/4 | Loss: 2.0442 | Temp: 0.0700 | img_emb: (4, 256) | txt_emb: (4, 256)
  [Epoch 1] Batch 4/4 | Loss: 1.4736 | Temp: 0.0700 | img_emb: (4, 256) | txt_emb: (4, 256)
>>> Epoch 1/3  Avg Loss: 1.6211

  [Epoch 2] Batch 1/4 | Loss: 1.5658 | Temp: 0.0700 | img_emb: (4, 256) | txt_emb: (4, 256)
  [Epoch 2] Batch 2/4 | Loss: 1.5296 | Temp: 0.0700 | img_emb: (4, 256) | txt_emb: (4, 256)
  [Epoch 2] Batch 3/4 | Loss: 1.5714 | Temp: 0.0700 | img_emb: (4, 256) | txt_emb: (4, 256)
  [Epoch 2] Batch 4/4 | Loss: 1.6958 | Temp: 0.0700 | img_emb: (4, 256) | txt_emb: (4, 256)
>>> Epoch 2/3  Avg Loss: 1.5907

  [Epoch 3] Batch 1/4 | Loss: 1.4413 | Temp: 0.0701 | img_emb: (4, 256) | txt_emb: (4, 256)
  [Epoch 3] Batch 2/4 | Loss: 1.5447 | Temp: 0.0701 | img_emb: (4, 256) | txt_emb: (

## Step 10 — Evaluation: Zero-Shot Retrieval

At inference time CLIP models are evaluated by **retrieval**: given an image, rank all candidate texts by cosine similarity and check if the correct one is top-k.

With dummy random data, R@1 ≈ 1/N is the random-chance baseline. This step verifies the evaluation code path, not actual performance.

In [11]:
@torch.no_grad()
def evaluate(
    model      : EchoCare_CLIP,
    dataloader : DataLoader,
    tokenizer  : CLIPTokenizer,
    device     : torch.device,
) -> tuple[torch.Tensor, float, float]:
    """
    Zero-shot image↔text retrieval evaluation.

    For each image in the dataset, rank all texts by cosine similarity and
    report Recall@1 in both directions.

    Returns
    -------
    sim_matrix : (N, N)  full pairwise similarity matrix
    i2t_r1     : float   image-to-text Recall@1  (fraction correct)
    t2i_r1     : float   text-to-image Recall@1
    """
    model.eval()
    all_image_embs = []
    all_text_embs  = []

    for images, texts in dataloader:
        images = images.to(device)
        input_ids, attention_mask = tokenize_texts(texts, tokenizer, device=device)

        # image_emb : (B, projection_dim)
        # text_emb  : (B, projection_dim)
        image_emb = model.encode_image(images)
        text_emb  = model.encode_text(input_ids, attention_mask)

        all_image_embs.append(image_emb.cpu())
        all_text_embs.append(text_emb.cpu())

    # Stack across batches
    all_image_embs = torch.cat(all_image_embs, dim=0)  # (N, projection_dim)
    all_text_embs  = torch.cat(all_text_embs,  dim=0)  # (N, projection_dim)

    # Cosine similarity matrix: (N, N)
    # Both are L2-normalised → dot product == cosine similarity
    sim_matrix = all_image_embs @ all_text_embs.T

    N      = all_image_embs.size(0)
    labels = torch.arange(N)

    # Recall@1: top-1 retrieved item matches the ground-truth paired item
    i2t_r1 = (sim_matrix.argmax(dim=1) == labels).float().mean().item()
    t2i_r1 = (sim_matrix.argmax(dim=0) == labels).float().mean().item()

    print(f"Evaluation results (N={N} samples)")
    print(f"  Image-to-Text R@1 : {i2t_r1 * 100:.1f}%")
    print(f"  Text-to-Image R@1 : {t2i_r1 * 100:.1f}%")
    print(f"  Random baseline   : {100.0 / N:.1f}%  (1/N)")
    print(f"  Similarity matrix shape: {sim_matrix.shape}")

    return sim_matrix, i2t_r1, t2i_r1


print("Running evaluation on dummy data...\n")
sim_matrix, i2t_r1, t2i_r1 = evaluate(model, dataloader, tokenizer, device)

Running evaluation on dummy data...

Evaluation results (N=16 samples)
  Image-to-Text R@1 : 6.2%
  Text-to-Image R@1 : 0.0%
  Random baseline   : 6.2%  (1/N)
  Similarity matrix shape: torch.Size([16, 16])


## Step 11 — End-to-End Shape Verification

Final sanity check that traces tensor shapes through every stage of the pipeline. This cell should run to completion without errors even when the EchoCare weights file is absent.

In [12]:
print("=" * 55)
print("End-to-End Shape Verification")
print("=" * 55)

model.eval()
B = 2  # mini-batch size for this check

with torch.no_grad():

    # ── 1. Raw inputs ──────────────────────────────────────────────────────
    dummy_images = torch.rand(B, 3, 256, 256).to(device)
    dummy_texts  = [
        "Echo showing left ventricular dysfunction.",
        "Normal right heart with trivial pericardial effusion.",
    ]
    print(f"\n[1] Input images         : {tuple(dummy_images.shape)}")
    #     Expected: (2, 3, 256, 256)

    # ── 2. SwinTransformer multi-scale features ────────────────────────────
    raw_feats = model.image_encoder.encoder(dummy_images)
    print(f"[2] SwinTransformer stages:")
    stage_channels = [128, 256, 512, 1024, 2048]
    stage_spatials = [128,  64,  32,   16,    8]
    for i, f in enumerate(raw_feats):
        print(
            f"     Stage {i}: {tuple(f.shape)}"
            f"  (expected (B, {stage_channels[i]}, {stage_spatials[i]}, {stage_spatials[i]}))"
        )

    # ── 3. Global Average Pooling ──────────────────────────────────────────
    deep   = raw_feats[-1]                # (B, 2048, 8, 8)
    pooled = deep.mean(dim=(2, 3))        # (B, 2048)
    print(f"[3] After GAP (deepest)  : {tuple(pooled.shape)}")  # (2, 2048)

    # ── 4. Full image embedding ────────────────────────────────────────────
    image_emb = model.encode_image(dummy_images)  # (B, projection_dim)
    print(f"[4] Image embedding      : {tuple(image_emb.shape)}")  # (2, 256)
    print(f"    L2 norms (should be 1.0): {image_emb.norm(dim=-1).tolist()}")

    # ── 5. Text tokenisation ───────────────────────────────────────────────
    input_ids, attn_mask = tokenize_texts(dummy_texts, tokenizer, device=device)
    print(f"[5] Token ids            : {tuple(input_ids.shape)}")   # (2, 77)
    print(f"    Attention mask       : {tuple(attn_mask.shape)}")   # (2, 77)

    # ── 6. Full text embedding ─────────────────────────────────────────────
    text_emb = model.encode_text(input_ids, attn_mask)  # (B, projection_dim)
    print(f"[6] Text embedding       : {tuple(text_emb.shape)}")   # (2, 256)
    print(f"    L2 norms (should be 1.0): {text_emb.norm(dim=-1).tolist()}")

    # ── 7. Similarity matrix ───────────────────────────────────────────────
    sim = image_emb @ text_emb.T  # (B, B)
    print(f"[7] Similarity matrix    : {tuple(sim.shape)}")  # (2, 2)
    print(f"    Values:\n{sim}")

    # ── 8. InfoNCE loss ────────────────────────────────────────────────────
    loss, _, _ = model(dummy_images, input_ids, attn_mask)
    print(f"[8] InfoNCE loss         : {loss.item():.4f}")
    #     With random init & B=2, expect ≈ log(2) ≈ 0.693

    # ── 9. Learnable temperature ───────────────────────────────────────────
    temp = model.log_temperature.exp().item()
    print(f"[9] Temperature          : {temp:.4f}")

print("\n" + "=" * 55)
print("All shapes verified — pipeline is functional.")
print("=" * 55)

End-to-End Shape Verification

[1] Input images         : (2, 3, 256, 256)
[2] SwinTransformer stages:
     Stage 0: (2, 128, 128, 128)  (expected (B, 128, 128, 128))
     Stage 1: (2, 256, 64, 64)  (expected (B, 256, 64, 64))
     Stage 2: (2, 512, 32, 32)  (expected (B, 512, 32, 32))
     Stage 3: (2, 1024, 16, 16)  (expected (B, 1024, 16, 16))
     Stage 4: (2, 2048, 8, 8)  (expected (B, 2048, 8, 8))
[3] After GAP (deepest)  : (2, 2048)
[4] Image embedding      : (2, 256)
    L2 norms (should be 1.0): [1.0, 1.0]
[5] Token ids            : (2, 77)
    Attention mask       : (2, 77)
[6] Text embedding       : (2, 256)
    L2 norms (should be 1.0): [1.0, 0.9999999403953552]
[7] Similarity matrix    : (2, 2)
    Values:
tensor([[0.0672, 0.0751],
        [0.0672, 0.0762]], device='cuda:0')
[8] InfoNCE loss         : 0.6903
[9] Temperature          : 0.0701

All shapes verified — pipeline is functional.


## Summary of Design Choices

| Component | Choice | Rationale |
|-----------|--------|-----------|
| Image backbone | EchoCare SwinTransformer (`feature_size=128`, `depths=[2,2,18,2]`, 2D, SwinV2) | Pretrained on echocardiography; exact same config ensures weight compatibility |
| Image feature extraction | Deepest stage (2048 ch) + GAP | Deepest features are most semantically abstract; GAP produces a fixed-size vector regardless of spatial size |
| Text backbone | CLIP `openai/clip-vit-base-patch32` text tower | Small (~63M), purpose-built for image–text alignment, strong zero-shot capabilities |
| Text backbone training | **Frozen** | Prevents forgetting of CLIP's alignment; projection head adapts to echo domain |
| Projection heads | 2-layer MLP (dim → dim → projection_dim) | Standard in contrastive learning (MoCo v3, CLIP); adds non-linearity for adaptation |
| Shared dim | `projection_dim=256` | Smaller than either encoder's native dim; forces compression into a compact shared space |
| Loss | Symmetric InfoNCE | Standard CLIP objective; both image→text and text→image objectives optimised simultaneously |
| Temperature | Learnable `log_temperature`, init `log(0.07)` | Allows the model to adjust sharpness of the similarity distribution during training |

## Next steps (when real data is ready)
1. Replace `DummyEchoDataset` with a real dataset that loads echo frames and paired clinical reports.
2. Add a cosine LR schedule with warmup.
3. Optionally unfreeze the CLIP text encoder after initial training convergence.
4. Evaluate zero-shot retrieval and downstream classification on held-out sets.
5. Consider using larger batches (≥256) to improve InfoNCE variance reduction.